In [16]:
import pandas as pd
import re
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd
import re

# 1. Recharger et Recalculer (Obligatoire pour éviter l'erreur)
df = pd.read_csv('donnees_propres_bi.csv', on_bad_lines='skip')
df['text_clean'] = (df['Titre'].fillna('') + " " + df['Description_Brute'].fillna('')).astype(str).str.lower()

# Définition des mots-clés (Rappel)
mots_info = ['java', 'python', 'technique', 'technicien', 'maintenance', 'industriel', 'dessin']
mots_diplome = ['bac+5', 'master', 'ingenieur', 'diplome', 'bts', 'dut', 'licence']
mots_exp = ['senior', 'expert', 'manager', 'responsable', 'chef', 'experience', 'annee']

def compter_mots(texte, liste_mots):
    score = 0
    for mot in liste_mots:
        if mot in texte: score += 1
    return score

print("Recalcul des scores en cours...")
df['Score_Info'] = df['text_clean'].apply(lambda x: compter_mots(x, mots_info))
df['Score_Diplome'] = df['text_clean'].apply(lambda x: compter_mots(x, mots_diplome))
df['Score_Exp'] = df['text_clean'].apply(lambda x: compter_mots(x, mots_exp))

# 2. Maintenant, l'analyse va marcher (car les colonnes existent !)
print("\n--- Statistiques des Scores ---")
print(df[['Score_Info', 'Score_Diplome', 'Score_Exp']].describe())

nb_diplome = len(df[df['Score_Diplome'] > 0])
nb_exp = len(df[df['Score_Exp'] > 0])

print(f"\nOffres avec mention de diplôme : {nb_diplome} sur {len(df)}")
print(f"Offres avec mention d'expérience : {nb_exp} sur {len(df)}")


# 2. Nettoyage Robuste (On vire les accents pour simplifier la recherche)
def nettoyer_texte(texte):
    if not isinstance(texte, str): return ""
    # Conversion en minuscules
    texte = texte.lower()
    # Suppression des accents (très important pour matcher 'ingénieur' avec 'ingenieur')
    texte = texte.replace('é', 'e').replace('è', 'e').replace('ê', 'e').replace('à', 'a').replace('ï', 'i')
    # On garde lettres et chiffres
    texte = re.sub(r'[^a-z0-9\s]', ' ', texte)
    return texte
# Vérification globale
print("--- Statistiques des Scores ---")
print(df[['Score_Info', 'Score_Diplome', 'Score_Exp']].describe())

# Vérification : Y a-t-il au moins UNE offre avec un diplôme ?
nb_diplome = len(df[df['Score_Diplome'] > 0])
nb_exp = len(df[df['Score_Exp'] > 0])

print(f"\nOffres avec mention de diplôme : {nb_diplome} sur {len(df)}")
print(f"Offres avec mention d'expérience : {nb_exp} sur {len(df)}")

# On combine Titre + Description
df['text_complet'] = df['Titre'].fillna('') + " " + df['Description_Brute'].fillna('')
df['text_clean'] = df['text_complet'].apply(nettoyer_texte)

# 3. LISTES DE MOTS-CLÉS (SANS ACCENTS)
# Enrichies pour attraper l'industrie, le dessin, le technique
mots_info_tech = [
    # Info
    'java', 'python', 'sql', 'data', 'code', 'developpeur', 'web', 'informatique', 'logiciel', 'it', 'digital', 'reseau',
    # Industrie / Technique (Nouveau !)
    'mecanique', 'electrique', 'electronique', 'maintenance', 'industriel', 'industrie', 'usine',
    'dessin', 'plan', 'conception', 'technique', 'technicien', 'ingenierie', 'automatisme', 'robotique',
    'support', 'systeme', 'machine', 'outil', 'production', 'atelier', 'batiment', 'chantier', 'essais', 'test'
]

mots_diplome = [
    'bac+5', 'master', 'ingenieur', 'diplome', 'ecole', 'superieur', 'universite', 'bac+3', 'licence',
    'bts', 'dut', 'formation', 'bac+2', 'certification', 'doctorat', 'phd'
]

mots_exp = [
    'senior', 'expert', 'confirme', 'experience', 'manager', 'directeur', 'responsable', 'chef', 'annee', 'annees',
    'gestion', 'piloter', 'autonomie', 'encadrement', 'equipe', 'management'
]

# 4. CALCUL DES SCORES (Méthode souple)
def compter_mots(texte, liste_mots):
    score = 0
    for mot in liste_mots:
        # On cherche si le mot est DANS le texte (ex: 'technicien' dans 'technicienne')
        if mot in texte:
            score += 1
    return score

print("Calcul des scores thématiques...")
df['Score_Info'] = df['text_clean'].apply(lambda x: compter_mots(x, mots_info_tech))
df['Score_Diplome'] = df['text_clean'].apply(lambda x: compter_mots(x, mots_diplome))
df['Score_Exp'] = df['text_clean'].apply(lambda x: compter_mots(x, mots_exp))

# 5. K-MEANS SUR LES SCORES
X = df[['Score_Info', 'Score_Diplome', 'Score_Exp']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# On garde k=4 profils types
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(X_scaled)
df['Cluster_ID'] = kmeans.labels_

# 6. ANALYSE ET CONSEILS (Seuils abaissés à 0.4)
print("\n>>> NOUVEAUX PROFILS (Moyennes) <<<")
print(df.groupby('Cluster_ID')[['Score_Info', 'Score_Diplome', 'Score_Exp']].mean())

def recommander_preparation(cluster_id):
    # On regarde les moyennes du cluster
    moyennes = df[df['Cluster_ID'] == cluster_id][['Score_Info', 'Score_Diplome', 'Score_Exp']].mean()

    conseils = []
    # Si le score moyen du cluster dépasse 0.4, on active le conseil
    if moyennes['Score_Info'] > 0.4:
        conseils.append("Misez sur vos compétences techniques (Outils/Terrain/Code)")
    if moyennes['Score_Diplome'] > 0.4:
        conseils.append("Valorisez votre diplôme")
    if moyennes['Score_Exp'] > 0.4:
        conseils.append("Mettez en avant votre expérience/management")

    if not conseils:
        return "Métier accessible : Misez sur votre motivation et soft skills !"

    return " + ".join(conseils)

df['Conseil_IA'] = df['Cluster_ID'].apply(recommander_preparation)

# 7. EXEMPLE DE RÉSULTAT
print("\nEXEMPLE DE RÉSULTAT CORRIGÉ :")
cols_to_show = ['Titre', 'Score_Info', 'Score_Diplome', 'Score_Exp', 'Conseil_IA']
# On affiche spécifiquement les lignes qui posaient problème avant (6, 9)
indices_interessants = [0, 1, 6, 7, 9]
# On filtre pour ne pas planter si l'index n'existe pas
indices_existants = [i for i in indices_interessants if i in df.index]
print(df.loc[indices_existants, cols_to_show])

# Sauvegarde
df.to_csv('donnees_finales_conseils.csv', index=False)


Recalcul des scores en cours...

--- Statistiques des Scores ---
         Score_Info  Score_Diplome     Score_Exp
count  15882.000000   15882.000000  15882.000000
mean       0.164589       0.003652      0.172208
std        0.467020       0.060323      0.386636
min        0.000000       0.000000      0.000000
25%        0.000000       0.000000      0.000000
50%        0.000000       0.000000      0.000000
75%        0.000000       0.000000      0.000000
max        4.000000       1.000000      2.000000

Offres avec mention de diplôme : 58 sur 15882
Offres avec mention d'expérience : 2680 sur 15882
--- Statistiques des Scores ---
         Score_Info  Score_Diplome     Score_Exp
count  15882.000000   15882.000000  15882.000000
mean       0.164589       0.003652      0.172208
std        0.467020       0.060323      0.386636
min        0.000000       0.000000      0.000000
25%        0.000000       0.000000      0.000000
50%        0.000000       0.000000      0.000000
75%        0.000000   